## E8 Colab notebook：Agent 與規則公平對照（#780，設計 `docs/evaluation/E8_DESIGN_V1.md`）

**這本 notebook 做什麼**：在 Colab GPU 上跑 E8 的本機模型推論（laya_text、laya_typed、Qwen2.5-VL-3B、Qwen2.5-VL-7B），零 API 費用。
- **第一部分（單點）**：116 個決策點 × 資訊級 L0–L4／L3F × 3 種候選順序；B1 的 T−1 快照（凍結）。只有第 7 格讀標籤表（D385 白名單）。
- **第二部分（閉迴路）**：B1 追人，模型只在交接點（LOST／SWITCH／AMBIGUOUS）回答；先跑 B1 重現閘門，**任一段不一致就停**。這部分在 Colab 沒有任何 GT，評分在 Mac。

**怎麼跑（使用者）**
1. Colab 選 **A100 GPU**（`執行階段 → 變更執行階段類型`）。
2. 把 `e8-colab-pack.zip` 和 `e8-colab-pack.zip.sha256` 放到 Google Drive 的 `MyDrive/amidst/`。
3. `執行階段 → 全部執行`（Run All）。中途斷線就重開再 Run All，已完成的組會從 `MyDrive/amidst/e8-outputs/checkpoints/` 續跑。
4. 跑完把 `MyDrive/amidst/e8-outputs/e8-run-<日期>.zip` 下載回 Mac，放到 `~/amidst/local/evidence/e8-run-<日期>/`。

**時間（§6.3）**：第 6 格先量每個 Qwen 模型前 20 筆的實際時間並重估整本時間；預估超過 4 h 時 Qwen 的 L0 只跑 `p0`，寫進 `RUN_NOTES.json` 的偏離。
**身分一律 UNCONFIRMED。** 不付費、不呼叫 API。


In [ ]:
# === CELL 1: CONFIGURATION (edit only here) ===
from pathlib import Path

GIT_REPO_URL = "https://github.com/Polalalabear/amidst.git"
PINNED_GIT_COMMIT = "62981f08ce4fc3a4ba329171fea7fb242474938f"  # exact 40-char SHA of claude/feat-e8; Cell 1B stops if the checkout differs
NOTEBOOK_VERSION_DATE = "2026-11-28-e8-r1"

PACK_ZIP = Path("/content/drive/MyDrive/amidst/e8-colab-pack.zip")
PACK_ROOT = Path("/content/e8-colab-pack")
OUTPUT_DIR = Path("/content/drive/MyDrive/amidst/e8-outputs")
REPO_DIR = Path("/content/amidst")

SMOKE_MODE = False          # True: deterministic stand-in backends, no GPU / no download (local CPU SMOKE, V7)
ACCELERATED = True          # GPU (bf16) when available

LAYA_COMMIT = "246177e5cb70334c9831c0730864a93c836aac8f"
MODELS = [
    {"model": "laya_text", "kind": "laya", "hf_repo_id": "convaiinnovations/laya", "hf_revision": LAYA_COMMIT, "checkpoint_subdir": ""},
    {"model": "laya_typed", "kind": "laya", "hf_repo_id": "convaiinnovations/laya", "hf_revision": LAYA_COMMIT, "checkpoint_subdir": "typed-decisions"},
    {"model": "qwen3b", "kind": "qwen", "hf_repo_id": "Qwen/Qwen2.5-VL-3B-Instruct", "hf_revision": "66285546d2b821cf421d4f5eb2576359d3770cd3"},
    {"model": "qwen7b", "kind": "qwen", "hf_repo_id": "Qwen/Qwen2.5-VL-7B-Instruct", "hf_revision": "cc594898137f460bfe9f0759e9844b3ce807cfb5"},
]
RUN_MODELS = None           # None = all four; or a list of model names
RUN_PART2 = True

PINNED_DEPS = ["torch==2.7.1", "torchvision==0.22.1", "transformers==4.53.3", "huggingface_hub==0.33.4", "safetensors==0.5.3", "accelerate==1.8.1"]
MIN_RUNTIME_VERSIONS = {"numpy": "2.0.0", "Pillow": "10.0.0", "matplotlib": "3.8.0"}
RUN_DEVIATIONS: list[str] = []   # free text, copied into RUN_NOTES.json


In [ ]:
# === CELL 1B: MOUNT GOOGLE DRIVE (COLAB) & CLONE / CHECKOUT PINNED REPO ===
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted Google Drive at /content/drive")
except ImportError:
    print("Not running in Google Colab; skipping drive.mount('/content/drive') (local fallback).")

if (Path.cwd() / "src" / "amidst").is_dir():
    REPO_DIR = Path.cwd()
    print(f"Already inside repo checkout: {REPO_DIR}; skipping git clone.")
elif not (REPO_DIR / ".git").is_dir():
    print(f"Cloning {GIT_REPO_URL} -> {REPO_DIR} ...")
    subprocess.check_call(["git", "clone", GIT_REPO_URL, str(REPO_DIR)])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])
else:
    print(f"Repo already present at {REPO_DIR}; checking out {PINNED_GIT_COMMIT} ...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])

# --- Version banner & pin check (Colab clone / checkout paths only; a local in-repo run just reports) ---
_head = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
_parties = [c["model"] for c in MODELS if RUN_MODELS is None or c["model"] in RUN_MODELS]
_skipped = []
print("=" * 72)
print(f"E8 NOTEBOOK {NOTEBOOK_VERSION_DATE} | repo commit {_head}")
print(f"RUNNING: {', '.join(_parties)} | SKIPPED (unaudited): {', '.join(_skipped) or 'none'}")
print("=" * 72)
if not (Path.cwd() / "src" / "amidst").is_dir() and _head != PINNED_GIT_COMMIT:
    raise RuntimeError(f"STOP: checked-out commit {_head} != PINNED_GIT_COMMIT {PINNED_GIT_COMMIT}")


In [ ]:
# === CELL 2: INSTALL PINNED DEPENDENCIES + MEMORY/DISK VERSION CONSISTENCY CHECK ===
import subprocess, sys

if (REPO_DIR / "src").is_dir() and str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from amidst.evaluation.e7_colab_runtime import assert_runtime_consistent, check_min_versions

if not SMOKE_MODE:
    print("Installing pinned dependencies:", PINNED_DEPS)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED_DEPS])
    # Stops with "重啟 runtime 後再 Run All" if torch/transformers/... were already imported at another version.
    RUNTIME_VERSIONS = assert_runtime_consistent(PINNED_DEPS)
    RUNTIME_VERSIONS["runtime_minimums"] = check_min_versions(MIN_RUNTIME_VERSIONS)
    print("Runtime versions consistent (memory == disk == pin):", RUNTIME_VERSIONS["disk"])
else:
    RUNTIME_VERSIONS = {"pins": {}, "disk": {}, "loaded": {}, "note": "SMOKE_MODE: pip install and version check skipped"}
    print("SMOKE_MODE=True; skipping pip install of PINNED_DEPS.")

In [ ]:
# === CELL 3: HARDWARE & ENVIRONMENT PROVENANCE ===
import json
from amidst.evaluation.e7_decider_eval import collect_hardware_provenance

HARDWARE_PROVENANCE = collect_hardware_provenance(REPO_DIR if REPO_DIR.is_dir() else None)
actual_dtype = HARDWARE_PROVENANCE.get("actual_accelerated_dtype", "fp32")
ACCELERATED_DTYPE = f"cuda_{actual_dtype}" if HARDWARE_PROVENANCE.get("gpu_name", "NONE") != "NONE" and actual_dtype != "fp32" else f"cpu_{actual_dtype}"
print(f"CUDA bf16 supported: {HARDWARE_PROVENANCE.get('cuda_bf16_supported')} | actual_accelerated_dtype: {actual_dtype} | label: {ACCELERATED_DTYPE}")
print(json.dumps(HARDWARE_PROVENANCE, indent=2, ensure_ascii=False))


In [ ]:
# === CELL 4: UNZIP UPLOAD PACK TO LOCAL DISK, THEN MANIFEST SHA-256 VERIFICATION (STOPS ON MISMATCH) ===
from amidst.evaluation.e7_colab_runtime import extract_pack_zip
from amidst.evaluation.e7_decider_eval import verify_manifest_sha256

PACK_ZIP_SHA256 = None
if PACK_ZIP.is_file():
    _ex = extract_pack_zip(PACK_ZIP, PACK_ROOT)  # checks the zip against <zip>.sha256, refuses path traversal
    PACK_ROOT = _ex["pack_root"]
    PACK_ZIP_SHA256 = _ex["zip_sha256"]
    print(f"Unzipped {PACK_ZIP.name} (sha256 {PACK_ZIP_SHA256}; checked against: {_ex['sha256_checked_against']}) -> {PACK_ROOT}")
elif (PACK_ROOT / "MANIFEST.sha256").is_file():
    print(f"No zip at {PACK_ZIP}; using already-extracted pack at {PACK_ROOT}.")
else:
    raise FileNotFoundError(f"PACK_ZIP_MISSING: {PACK_ZIP} (upload e8-colab-pack.zip and its .sha256 to Drive)")

MANIFEST_VERIFICATION = verify_manifest_sha256(PACK_ROOT)
print("MANIFEST verified:", MANIFEST_VERIFICATION)

In [ ]:
# === CELL 5: LOAD E8 REQUESTS (INFERENCE SIDE: NO ANSWER TABLE) + GT KEY SCAN + G5 ===
import json, sys
for _p in (REPO_DIR / "src", REPO_DIR / "scripts" / "evaluation"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
from amidst.evaluation import e8_colab_run as RUN
from amidst.evaluation import e8_requests as R
from amidst.live_mode.aopen.arms import reject_forbidden

PACK = RUN.load_pack(PACK_ROOT)   # raises on G5 (decision points != requests' base ids)
for _r in PACK["requests"]:
    reject_forbidden(_r, _r["request_id"])
print(f"{len(PACK['requests'])} requests, {PACK['info']['decision_points']} decision points, {len(PACK['cases'])} cases; GT key scan PASS")


In [ ]:
# === CELL 6: PART 1 INFERENCE (RESUMABLE per model x level x order; 4 h RULE; NO GT) ===
import gc, statistics
from amidst.evaluation.e7_colab_runtime import run_fingerprint
from amidst.evaluation.e7_decider_eval import build_laya_native_inference_fn, build_qwen_vl_inference_fn, resolve_hf_provenance

FINGERPRINT = run_fingerprint(git_commit=HARDWARE_PROVENANCE.get("git_commit", "UNKNOWN"), dataset_scope="e8", n_requests=len(PACK["requests"]),
                              smoke_mode=SMOKE_MODE, cpu_sample_cap=None, pinned_deps=PINNED_DEPS,
                              pack_manifest_sha256=MANIFEST_VERIFICATION.get("manifest_sha256") if isinstance(MANIFEST_VERIFICATION, dict) else None)
HF_PROVENANCE, TIMING, COMPLETED_GROUPS, RESUMED_GROUPS, PART1_RESULTS = {}, {}, [], [], []

def load_backend(cfg):
    """-> (raw callable, infer fn or None). SMOKE_MODE: deterministic stand-ins."""
    if SMOKE_MODE:
        return (RUN.smoke_laya if cfg["kind"] == "laya" else RUN.smoke_vlm), None
    if cfg["kind"] == "laya":
        fn = build_laya_native_inference_fn(cfg["hf_repo_id"], cfg["hf_revision"], custom_code_audited=True, audited_commit=LAYA_COMMIT,
                                            checkpoint_subdir=cfg["checkpoint_subdir"])
        return (lambda state, q: fn.raw(state, q, ACCELERATED)), fn
    fn = build_qwen_vl_inference_fn(cfg["hf_repo_id"], cfg["hf_revision"])
    return (lambda system, text, images, n: fn.raw(system, text, images, n, ACCELERATED)), fn

def free(fn):
    del fn
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

def ask_for(cfg, raw):
    return RUN.answer_fn(cfg["model"], laya_raw=raw if cfg["kind"] == "laya" else None, vlm_raw=raw if cfg["kind"] == "qwen" else None, pack=PACK)

# 1) timing (design §6.3): first 20 requests per Qwen kind, then the 4 h rule
QWEN_L0_P0_ONLY = False
for cfg in [c for c in MODELS if c["kind"] == "qwen" and (RUN_MODELS is None or c["model"] in RUN_MODELS)]:
    raw, fn = load_backend(cfg)
    ask = ask_for(cfg, raw)
    sample = {k: [r for r in PACK["requests"] if r["level"] == lv and r["perm"] == "p0"] for k, lv in (("L0", "L0"), ("text", "L2"), ("L4", "L4"), ("L3F", "L3F"))}
    TIMING[cfg["model"]] = {k: RUN.timing_sample(v, ask) for k, v in sample.items() if v and (k != "L3F" or cfg["model"] == "qwen7b")}
    free(fn)
PROJECTED_H = RUN.projected_hours(PACK["requests"], TIMING, part2_cases=len(PACK["cases"]))
if PROJECTED_H > RUN.TIME_LIMIT_H and not SMOKE_MODE:
    QWEN_L0_P0_ONLY = True
    RUN_DEVIATIONS.append(f"4h rule: projected {PROJECTED_H:.2f} h > {RUN.TIME_LIMIT_H} h on this GPU; Qwen L0 runs p0 only (design §6.3)")
print("Measured medians (s):", TIMING, "| projected hours:", round(PROJECTED_H, 2), "| Qwen L0 p0 only:", QWEN_L0_P0_ONLY)

# 2) inference
for cfg in [c for c in MODELS if RUN_MODELS is None or c["model"] in RUN_MODELS]:
    if not SMOKE_MODE:
        HF_PROVENANCE[cfg["model"]] = resolve_hf_provenance(cfg["hf_repo_id"], cfg["hf_revision"])
    raw, fn = load_backend(cfg)
    ask = ask_for(cfg, raw)
    audit = RUN.laya_audit_fn(fn, cfg["model"]) if (fn is not None and cfg["kind"] == "laya") else None
    for lv, perm, reqs in RUN.part1_groups(PACK["requests"], cfg["model"], qwen_l0_p0_only=QWEN_L0_P0_ONLY):
        res, resumed = RUN.run_group(cfg["model"], lv, perm, reqs, ask, OUTPUT_DIR, FINGERPRINT, audit=audit if lv != "L0" else None)
        (RESUMED_GROUPS if resumed else COMPLETED_GROUPS).append(f"{cfg['model']}/{lv}/{perm}")
        PART1_RESULTS.append(res)
        print(f"{cfg['model']:10s} {lv:4s} {perm} n={res['n']} p50={res['latency_p50_s']} invalid={res['format_invalid']}{' (resumed)' if resumed else ''}")
    free(fn)


In [ ]:
# === CELL 7: PART 1 SCORING — THE ONLY CELL THAT READS THE LABEL-ONLY ANSWER TABLE (D385) ===
# Provisional: the table cannot separate TRUE_DEPARTURE from FULL_BLIND; the Mac command recomputes with GT and is final.
from amidst.evaluation import e8_score as S

_table = json.loads((PACK_ROOT / "scoring_only" / "answer_table.json").read_text())
_regime = {b: S.regime_from_label(e) for b, e in _table.items()}
_family = {r["base_id"]: PACK["cases"][r["case_id"]]["family"] for r in PACK["requests"]}
PART1_SUMMARY = S.summary([rec for res in PART1_RESULTS for rec in res["records"]], PACK["requests"], _table, _regime, _family)
SELECTION = PART1_SUMMARY["selection"]   # aggregated choice only (which k* per model); no label enters any request
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "e8_part1_summary_colab.json").write_text(json.dumps(PART1_SUMMARY, indent=1, sort_keys=True, default=str))
for k, v in PART1_SUMMARY["accuracy"].items():
    print(f"{k:22s} majority={v['majority_acc']} mean={v['mean_acc']} n={v['n']} needs_mac={v['needs_mac']}")
print("Provisional H1:", {h: v["status"] for h, v in PART1_SUMMARY["primary"].items() if not h.startswith("_")})
print("Selection:", SELECTION)


In [ ]:
# === CELL 8: PART 2 CLOSED LOOP (B1 GATE FIRST; RESUMABLE per group; NO GT ON COLAB) ===
PART2_RESULTS, B1_GATE = [], {}
if RUN_PART2:
    _graph = json.loads((REPO_DIR / "configs/q1q5/graphs/r004/camera_graph_v3.json").read_text())
    B1_GATE = RUN.b1_gate(PACK, _graph)
    _bad = {c: g for c, g in B1_GATE.items() if not g["ok"]}
    if _bad:
        raise RuntimeError(f"G1/V2 STOP: B1 not reproduced on {sorted(_bad)} -> report to the coordinator; do not skip cases")
    print(f"B1 reproduction gate: {len(B1_GATE)}/{len(B1_GATE)} cases identical")
    groups = [(f"B1+{p}", RUN.baseline_ask(p), R.BASELINE_LEVEL[p], "ZERO_LATENCY", None) for p in RUN.PART2_RULE_GROUPS]
    for cfg in [c for c in MODELS if RUN_MODELS is None or c["model"] in RUN_MODELS]:
        k = (SELECTION["per_model"].get(cfg["model"]) or {}).get("level")
        if k is None:
            RUN_DEVIATIONS.append(f"part2: {cfg['model']} has no scored text level; skipped")
            continue
        latencies = ["ZERO_LATENCY"] + (["MEASURED"] if (SELECTION.get("selected") or {}).get("model") == cfg["model"] else [])
        for lat in latencies:
            groups.append((f"B1+{cfg['model']}@{k}" + ("" if lat == "ZERO_LATENCY" else "+MEASURED"), None, k, lat, cfg))
    for name, ask, level, lat, cfg in groups:
        fn = None
        if cfg is not None:
            raw, fn = load_backend(cfg)
            ask = ask_for(cfg, raw)
        res, resumed = RUN.run_part2_group(PACK, _graph, name, ask, level, lat, OUTPUT_DIR, FINGERPRINT)
        (RESUMED_GROUPS if resumed else COMPLETED_GROUPS).append(f"part2/{name}/{lat}")
        PART2_RESULTS.append({k: v for k, v in res.items() if k != "blogs"})
        print(f"{name:32s} {lat:12s} asks={res['asks']}{' (resumed)' if resumed else ''}")
        if fn is not None:
            free(fn)


In [ ]:
# === CELL 9: PACKAGE RUN (checkpoints + summaries + RUN_NOTES.json -> one zip on Drive) ===
import datetime as _dt
from amidst.evaluation.e7_colab_runtime import build_run_notes, package_run

RUN_NOTES = build_run_notes(notebook_version=NOTEBOOK_VERSION_DATE, git_commit=HARDWARE_PROVENANCE.get("git_commit", "UNKNOWN"),
                            python_version=HARDWARE_PROVENANCE.get("python_version", "UNKNOWN"), pinned_deps=PINNED_DEPS,
                            versions=RUNTIME_VERSIONS, pack_zip_sha256=PACK_ZIP_SHA256, manifest_verification=MANIFEST_VERIFICATION,
                            smoke_mode=SMOKE_MODE, completed_groups=COMPLETED_GROUPS, resumed_groups=RESUMED_GROUPS, skipped_models=[],
                            hf_provenance=HF_PROVENANCE, deviations=RUN_DEVIATIONS)
RUN_NOTES.update({"schema_version": "e8_run_notes_v1", "experiment": "E8", "timing_medians_s": TIMING, "projected_hours": PROJECTED_H,
                  "qwen_l0_p0_only": QWEN_L0_P0_ONLY, "hardware": HARDWARE_PROVENANCE, "b1_gate": B1_GATE, "part2_groups": PART2_RESULTS,
                  "pack_info": PACK["info"]})
_tag = _dt.datetime.now(_dt.timezone.utc).strftime("%Y%m%d")
PACKAGED = package_run(OUTPUT_DIR, f"e8-run-{_tag}", RUN_NOTES, extra_files=[OUTPUT_DIR / "e8_part1_summary_colab.json"])
print("Packaged run:", PACKAGED)
print("下載這個 zip 回 Mac：", PACKAGED["zip"])
